# Ayudantía Nº6:
### **Análisis de datos numéricos:** Medidas descriptivas, comparaciones y gráficos con `ggplot2`


## Antes de comenzar ...


1. Sube el archivo **`base_antropologia_limpia.xlsx`** desde el panel **Archivos**
2. Ejecuta las celdas en orden. La importación se realiza directamente con `openxlsx::read.xlsx()`

- Abre este `.ipynb` desde **Archivo → Subir cuaderno** en [Google Colab](https://colab.research.google.com/)
- Selecciona **R** en **Entorno de ejecución → Cambiar tipo de entorno de ejecución**
- En el panel lateral **Archivos**, sube `base_antropologia_limpia.xlsx` (con ese nombre exacto). En Colab debe quedar en `/content/`.
- Ejecuta las celdas en orden, desde la instalación de paquetes hasta los ejercicios

Se utilizan las variables numéricas originales `edad` y `ea_04_notas_ultimo_semestre`. Las variables agrupadas se distinguen de las numéricas

In [ ]:
# Instalar pacman solo si no está disponible
if (!requireNamespace("pacman", quietly = TRUE)) {
  install.packages("pacman", repos = "https://cloud.r-project.org")
}

# Paquetes que usaremos en esta ayudantía
pacman::p_load(tidyverse, openxlsx, scales, DescTools)

In [ ]:
# Importar directamente el archivo Excel en R
# Primero, sube base_antropologia_limpia.xlsx al panel Archivos de Colab.
ruta_base <- "/content/base_antropologia_limpia.xlsx"
if (!file.exists(ruta_base)) {
  stop("No se encontró base_antropologia_limpia.xlsx. Súbela al panel Archivos de Colab con ese nombre exacto.")
}
base <- openxlsx::read.xlsx(ruta_base)

# Comprobar que la base se cargó correctamente
dim(base)
names(base)


### 1. ¿Qué variables podemos analizar numéricamente?

- `edad`: edad en años. Es numérica; podemos calcular media, mediana, dispersión y cuantiles
- `ea_04_notas_ultimo_semestre`: promedio de notas del último semestre. Es numérica; también permite cálculos descriptivos
- `identidad_genero_r`, `ea_03_descripcion_carga_academica` y `ea_05_satisfaccion_rendimiento_academico_r`: son categóricas. **No calcularemos promedios de estas categorías**; las utilizaremos para comparar las edades o las notas de distintos grupos

**Práctica de lectura:** ¿por qué no corresponde calcular el promedio de `edad_r`? ¿Qué información se pierde cuando la edad se presenta en intervalos?

In [ ]:
# Examinar el tipo de dato y los primeros valores
class(base$edad)
class(base$ea_04_notas_ultimo_semestre)
head(base$edad)
head(base$ea_04_notas_ultimo_semestre)

# Confirmar categorías reales de una variable de agrupación
table(base$ea_05_satisfaccion_rendimiento_academico_r, useNA = "ifany")

### 2. Casos válidos: ¿por qué no siempre usamos las 147 respuestas?

`is.na()` identifica valores ausentes. `sum(!is.na(...))` cuenta los registros que sí tienen un dato disponible. `na.rm = TRUE` indica a las funciones estadísticas que ignoren los valores ausentes **en ese cálculo**, sin eliminarlos de la base.

**Práctica:** compara los casos válidos de edad y notas. ¿Por qué es importante indicar el número de observaciones al presentar un promedio?

In [ ]:
# Número de casos válidos y ausentes
sum(!is.na(base$edad))
sum(is.na(base$edad))
sum(!is.na(base$ea_04_notas_ultimo_semestre))
sum(is.na(base$ea_04_notas_ultimo_semestre))

Es importante indicar el número de observaciones (n) al presentar un promedio porque permite conocer cuántos casos fueron considerados en su cálculo y evaluar qué tan representativo puede ser de los datos analizados.

Un promedio calculado con pocas observaciones puede ser más sensible a valores extremos y presentar mayor variabilidad que uno calculado con una cantidad mayor de casos.

## 3. Medidas de tendencia central: media y mediana

Sabemos que la **media** es el promedio aritmético y la **mediana** es el valor que divide los datos ordenados en dos mitades. Cuando hay valores especialmente altos o bajos, la media puede desplazarse más que la mediana.

En la última clase estas medidas se aplican a ingresos. Aquí las aplicaremos a **edades y notas reales** para ver cómo cambia su interpretación según la variable.

In [ ]:
# R base: media y mediana de edad
mean(base$edad, na.rm = TRUE)
median(base$edad, na.rm = TRUE)

# El mismo cálculo con tidyverse
base %>%
  summarise(
    media_edad = mean(edad, na.rm = TRUE),
    mediana_edad = median(edad, na.rm = TRUE)
  )

### Ejercicio 1:

**Completa el código.** Calcula la media y la mediana de `ea_04_notas_ultimo_semestre` utilizando `summarise()`. Sustituye los espacios indicados en el comentario. Después explica en palabras qué significa cada resultado.

**Pregunta:** ¿por qué es incorrecto decir que todas las personas obtuvieron exactamente el promedio de notas?

In [ ]:
# Completa en una línea nueva tomando como modelo el código anterior:
# base %>%
#   summarise(
#     media_notas = ______(ea_04_notas_ultimo_semestre, na.rm = TRUE),
#     mediana_notas = ______(ea_04_notas_ultimo_semestre, na.rm = TRUE)
#   )

<details><summary>Ver una solución posible del ejercicio 1</summary>

```r
base %>%
  summarise(
    media_notas = mean(ea_04_notas_ultimo_semestre, na.rm = TRUE),
    mediana_notas = median(ea_04_notas_ultimo_semestre, na.rm = TRUE)
  )
```

La media resume el promedio de las notas disponibles; la mediana identifica el valor central. Ninguna implica que todas las personas tengan ese valor.
</details>

## 4. Medidas de dispersión: rango, varianza y desviación estándar

Dos grupos pueden tener el mismo promedio, pero valores más o menos dispersos. La **desviación estándar** (`sd()`) expresa la dispersión en las mismas unidades de la variable (años o puntos de nota). La **varianza** (`var()`) expresa la dispersión en unidades al cuadrado, por lo que suele ser menos intuitiva.

La desviación estándar **no es literalmente el promedio de las distancias absolutas**: se calcula a partir de desviaciones cuadráticas respecto de la media.

In [ ]:
# Dispersión de las edades
min(base$edad, na.rm = TRUE)
max(base$edad, na.rm = TRUE)
range(base$edad, na.rm = TRUE)
var(base$edad, na.rm = TRUE)
sd(base$edad, na.rm = TRUE)

# Verificar la relación entre varianza y desviación estándar
sqrt(var(base$edad, na.rm = TRUE))

### Ejercicio 2:

**Modifica el código.** Cambia `edad` por `ea_04_notas_ultimo_semestre` en las funciones `min()`, `max()` y `sd()`. ¿En qué unidades se expresa cada resultado? ¿Por qué no sería adecuado comparar directamente una desviación estándar de 3 años con otra de 0,5 puntos de nota para decidir cuál variable es "más variable"?

In [ ]:
# Escribe aquí tus tres cálculos. Modelo:
# min(base$ea_04_notas_ultimo_semestre, na.rm = TRUE)
# max(base$_________________________, na.rm = TRUE)
# sd(base$_________________________, na.rm = TRUE)

No es adecuado comparar directamente ambas desviaciones estándar porque las variables están medidas en **unidades diferentes**: la edad se expresa en años y las calificaciones en puntos de nota. Por lo tanto, una desviación estándar de 3 años no significa necesariamente que la edad presente mayor variabilidad que las notas, cuya desviación estándar es de 0,5 puntos

#### Las desviaciones estándar deben interpretarse considerando la unidad y la escala de cada variable. Una desviación estándar numéricamente mayor no implica necesariamente una mayor variabilidad relativa

## 5. Resumen descriptivo completo con `summarise()`

Una sola tabla puede reunir **casos válidos, media, mediana, desviación estándar, mínimo y máximo**. Esto permite describir la distribución sin mirar cada respuesta por separado.

**Observa:** `n()` cuenta filas del grupo, mientras que `sum(!is.na(...))` cuenta respuestas válidas de la variable específica.

In [ ]:
# Resumen de las notas del último semestre
base %>%
  summarise(
    casos_validos = sum(!is.na(ea_04_notas_ultimo_semestre)),
    media = mean(ea_04_notas_ultimo_semestre, na.rm = TRUE),
    mediana = median(ea_04_notas_ultimo_semestre, na.rm = TRUE),
    desviacion = sd(ea_04_notas_ultimo_semestre, na.rm = TRUE),
    minimo = min(ea_04_notas_ultimo_semestre, na.rm = TRUE),
    maximo = max(ea_04_notas_ultimo_semestre, na.rm = TRUE)
  )

### Ejercicio 3:

**Comprende el código.** ¿Qué pasaría si quitáramos `na.rm = TRUE` del cálculo de la media de notas? ¿Por qué `casos_validos` es menor que el número total de filas? Repite el resumen para `edad` cambiando únicamente el nombre de la variable.

- No todas las personas encuestadas proporcionaron información sobre sus notas. La base contiene 147 observaciones, pero solo 116 tienen datos válidos en esta variable. Por lo tanto, existen 31 valores faltantes que no se consideran al calcular el promedio.

## 6. Cuartiles: mirar cómo se distribuyen los valores

Los cuartiles separan la distribución ordenada en cuatro partes. **Q1** es el percentil 25; **Q2** coincide con la mediana; **Q3** es el percentil 75. El rango intercuartílico (`IQR()`) indica la amplitud del 50 % central de los datos.

Los cuartiles permiten comprender los gráficos de caja que veremos después.

In [ ]:
# Cuartiles de las notas
quantile(base$ea_04_notas_ultimo_semestre, probs = seq(0, 1, 0.25), na.rm = TRUE)
IQR(base$ea_04_notas_ultimo_semestre, na.rm = TRUE)

# Cuartiles de la edad
quantile(base$edad, probs = seq(0, 1, 0.25), na.rm = TRUE)

### Ejercicio 4:

**Interpreta.** ¿Qué significa que el cuartil 1 de las notas sea un determinado valor? ¿Por qué el 50 % central no necesariamente contiene el 50 % exacto de observaciones cuando hay muchos valores repetidos?

## 7. Comparar grupos: notas según satisfacción académica

**Pregunta de investigación:** ¿las notas del último semestre son similares entre quienes declaran estar satisfechos e insatisfechos con su rendimiento?

La variable numérica será `ea_04_notas_ultimo_semestre` y la variable que define los grupos será `ea_05_satisfaccion_rendimiento_academico_r`.

Este cruce es descriptivo: observar diferencias entre grupos **no demuestra que la satisfacción cause un cambio en las notas**, ni viceversa.

In [ ]:
# Confirmar los nombres de las categorías, sin inventarlas
table(base$ea_05_satisfaccion_rendimiento_academico_r, useNA = "ifany")

# Tabla descriptiva por satisfacción académica
tabla_notas_satisfaccion <- base %>%
  filter(!is.na(ea_04_notas_ultimo_semestre),
         !is.na(ea_05_satisfaccion_rendimiento_academico_r)) %>%
  group_by(ea_05_satisfaccion_rendimiento_academico_r) %>%
  summarise(
    n = n(),
    media = mean(ea_04_notas_ultimo_semestre),
    mediana = median(ea_04_notas_ultimo_semestre),
    desviacion = sd(ea_04_notas_ultimo_semestre),
    .groups = "drop"
  )

tabla_notas_satisfaccion

### Ejercicio 5:

**Cambia una instrucción.** Añade `%>% arrange(desc(media))` al final de la tabla anterior. ¿Qué grupo aparece primero? ¿Qué cambia: los valores calculados o solamente el orden de las filas? ¿Cuántas personas forman cada grupo?

**Para interpretar:** utiliza tanto la media como el número de casos válidos. No confundas «promedio de notas más alto» con «mayor número de estudiantes».

In [ ]:
# Prueba aquí el orden descendente de la media:
tabla_notas_satisfaccion %>%
  arrange(desc(media))

## 8. Histograma: ¿cómo se distribuyen las edades?

Un **histograma** representa una variable numérica mediante intervalos. A diferencia de un gráfico de barras categórico, las barras corresponden a **rangos contiguos de valores numéricos**.

`binwidth` controla la amplitud de cada intervalo; cambiarlo puede modificar la apariencia de la distribución sin modificar los datos.

In [ ]:
ggplot(base, aes(x = edad)) +
  geom_histogram(binwidth = 1, fill = "steelblue", color = "white") +
  labs(title = "Distribución de las edades del estudiantado",
       x = "Edad (años)", y = "Número de estudiantes") +
  theme_minimal()

### Ejercicio 6:

**Cambia colores e intervalos.** Modifica `fill = "steelblue"` por `"darkorange"` y después por `"seagreen"`. Cambia `binwidth = 1` por `binwidth = 2` y luego `binwidth = 3`.

**Responde:** ¿cuál versión permite identificar mejor las edades más frecuentes? ¿Cambian las edades reales cuando modificas `binwidth`? ¿Se observan edades más alejadas del grupo central?

In [ ]:
# Una solución posible: intervalos de dos años y color naranja
ggplot(base, aes(x = edad)) +
  geom_histogram(binwidth = 2, fill = "darkorange", color = "white") +
  labs(title = "Distribución de edades: intervalos de 2 años",
       x = "Edad (años)", y = "Frecuencia") +
  theme_classic()

## 9. Histograma de notas: cuidado con el ancho de intervalo

Las notas numéricas tienen una escala distinta de la edad. Por eso, un intervalo de **1 punto de nota** sería demasiado amplio para observar diferencias finas. Probaremos intervalos de `0.2` y `0.5` puntos, conservando la escala original de las notas.

In [ ]:
ggplot(base, aes(x = ea_04_notas_ultimo_semestre)) +
  geom_histogram(binwidth = 0.2, fill = "mediumpurple3", color = "white") +
  labs(title = "Distribución de las notas del último semestre",
       x = "Promedio de notas", y = "Número de estudiantes") +
  theme_minimal()

### Ejercicio 7:

**Compara.** Cambia `binwidth = 0.2` por `binwidth = 0.5`. ¿En cuál histograma se aprecian más detalles? ¿En cuál se simplifica más la distribución? Explica por qué **no usarías automáticamente el mismo `binwidth` para edad y notas**.

## 10. Diagrama de caja: mediana, cuartiles y posibles valores atípicos

El **boxplot** resume la mediana, los cuartiles y la dispersión. Los puntos fuera de los bigotes representan posibles valores atípicos según la regla del gráfico; **no son necesariamente errores** y no debemos eliminarlos automáticamente.

En el boxplot estándar de `ggplot2`, la línea dentro de la caja es la **mediana**, no la media. Los bigotes tampoco representan siempre el mínimo y el máximo.

In [ ]:
ggplot(base, aes(y = edad)) +
  geom_boxplot(fill = "skyblue", color = "grey30", width = 0.4) +
  labs(title = "Diagrama de caja de la edad",
       x = NULL, y = "Edad (años)") +
  theme_minimal()

### Ejercicio 8:

**Modifica y explica.** Cambia `fill = "skyblue"` por `"tomato"`, y `width = 0.4` por `width = 0.7`. ¿Qué cambia visualmente? ¿Se modifica la mediana? ¿Qué significan los puntos que aparecen fuera de los bigotes?

In [ ]:
# Ejemplo resuelto con otro color y otro ancho
ggplot(base, aes(y = edad)) +
  geom_boxplot(fill = "tomato", color = "grey20", width = 0.7) +
  labs(title = "Edades: mediana y dispersión",
       x = NULL, y = "Edad (años)") +
  theme_classic()

## 11. Boxplot comparativo: notas según satisfacción

El histograma nos mostró la distribución general; ahora el boxplot permite **comparar distribuciones de notas entre grupos**. Recuerda: la variable del eje vertical sigue siendo numérica y el eje horizontal indica los grupos.

Esta comparación se basa únicamente en los casos que tienen **nota y satisfacción disponibles**.

In [ ]:
ggplot(
  base %>% filter(!is.na(ea_04_notas_ultimo_semestre),
                  !is.na(ea_05_satisfaccion_rendimiento_academico_r)),
  aes(x = ea_05_satisfaccion_rendimiento_academico_r,
      y = ea_04_notas_ultimo_semestre,
      fill = ea_05_satisfaccion_rendimiento_academico_r)
) +
  geom_boxplot(width = 0.6, alpha = 0.75) +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "Notas según satisfacción con el rendimiento",
       x = "Satisfacción académica", y = "Promedio de notas") +
  guides(fill = "none") +
  theme_minimal()

### Ejercicio 9:

**Cambia el gráfico.** Reemplaza `palette = "Set2"` por `"Pastel1"`, después cambia `width = 0.6` por `0.4` y añade `coord_flip()` al final.

**Interpretación:** ¿qué grupo presenta una mediana más alta? ¿Cuál parece tener mayor dispersión? ¿Qué información aporta el boxplot que no aparece en una tabla con solo la media?

In [ ]:
# Variante con otra paleta y barras horizontales
ggplot(
  base %>% filter(!is.na(ea_04_notas_ultimo_semestre),
                  !is.na(ea_05_satisfaccion_rendimiento_academico_r)),
  aes(x = ea_05_satisfaccion_rendimiento_academico_r,
      y = ea_04_notas_ultimo_semestre,
      fill = ea_05_satisfaccion_rendimiento_academico_r)
) +
  geom_boxplot(width = 0.4, alpha = 0.8) +
  scale_fill_brewer(palette = "Pastel1") +
  coord_flip() +
  guides(fill = "none") +
  labs(title = "Distribución de notas por satisfacción académica",
       x = "Satisfacción", y = "Promedio de notas") +
  theme_classic()

## 12. Otra comparación: edades según carga académica percibida

La carga académica es una variable **ordinal**: *Ligera → Moderada → Pesada → Muy pesada*. Para representar la edad por grupos, indicaremos ese orden en el gráfico **sin modificar los valores originales de la base**.

**Pregunta:** ¿los estudiantes que perciben una carga más pesada son necesariamente mayores? La tabla y el gráfico permiten explorar esta relación, pero no demostrar causalidad.

In [ ]:
# Resumen numérico de edad según carga académica
base %>%
  group_by(ea_03_descripcion_carga_academica) %>%
  summarise(
    n = sum(!is.na(edad)),
    media_edad = mean(edad, na.rm = TRUE),
    mediana_edad = median(edad, na.rm = TRUE),
    desviacion_edad = sd(edad, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(media_edad))

In [ ]:
# El orden de las categorías se define solo para esta visualización
ggplot(base,
       aes(x = factor(ea_03_descripcion_carga_academica,
                      levels = c("Ligera", "Moderada", "Pesada", "Muy pesada")),
           y = edad)) +
  geom_boxplot(fill = "seagreen3", width = 0.6) +
  labs(title = "Distribución de edades según carga académica percibida",
       x = "Carga académica", y = "Edad (años)") +
  theme_minimal()

### Ejercicio 10:

**Reflexiona sobre tamaños de grupo.** ¿Qué grupo tiene más estudiantes? ¿Cuál tiene menos? ¿Es igual de estable un resumen basado en 10 observaciones que uno basado en 66? ¿Por qué no sería correcto afirmar que la carga académica modifica la edad?

*Las diferencias entre grupos permiten identificar patrones descriptivos, pero no demuestran que una variable sea la causa de otra.*

## 13. Coeficiente de Gini: una aplicación crítica

La Clase 9 introduce el coeficiente de Gini para estudiar **desigualdad en distribuciones numéricas**, especialmente ingresos. Nuestra encuesta no contiene una variable equivalente de ingresos individuales comparable con la CASEN. Podemos calcular el Gini de edad **solo como ejercicio técnico**, pero **no debemos interpretarlo como desigualdad socioeconómica** ni trasladar automáticamente umbrales usados para ingresos.

Este apartado es opcional y sirve para distinguir entre **saber ejecutar una función** y **saber si tiene sentido sustantivo utilizarla**.

In [ ]:
# Aplicación ilustrativa a la distribución de edades
DescTools::Gini(base$edad, na.rm = TRUE)

### Ejercicio 11:

**Discusión metodológica.** ¿Por qué el Gini de edades no permite afirmar que exista mayor o menor desigualdad económica entre estudiantes? ¿Qué tipo de variable sería más apropiada para estudiar desigualdad de ingresos?

## 14. Desafío integrador: describe, grafica e interpreta

### Ejercicio 12:

**Pregunta:** ¿Cómo se distribuyen las notas del último semestre entre estudiantes satisfechos e insatisfechos con su rendimiento?

**Tu trabajo:**

1. Comprueba cuántos casos tienen nota y satisfacción disponibles.
2. Construye una tabla con `n`, `mean()`, `median()` y `sd()` por satisfacción.
3. Construye un `geom_boxplot()` con colores de tu elección.
4. Cambia el título y los nombres de los ejes usando `labs()`.
5. Escribe un párrafo de 4–5 líneas que compare **medianas, dispersión y tamaños de grupo**, sin afirmar causalidad.

Puedes reutilizar y adaptar el código de los apartados 7 y 11. No necesitas crear datos nuevos ni recodificar categorías.

In [ ]:
# Espacio para tu solución del desafío integrador
# Pista para la tabla:
# base %>%
#   filter(!is.na(ea_04_notas_ultimo_semestre),
#          !is.na(ea_05_satisfaccion_rendimiento_academico_r)) %>%
#   group_by(ea_05_satisfaccion_rendimiento_academico_r) %>%
#   summarise(n = n(), media = mean(____), mediana = median(____),
#             desviacion = sd(____), .groups = "drop")

# Pista para el gráfico:
# ggplot(base, aes(x = ea_05_satisfaccion_rendimiento_academico_r,
#                  y = ea_04_notas_ultimo_semestre)) +
#   geom_boxplot(fill = "______") +
#   labs(title = "______", x = "______", y = "______") +
#   theme_minimal()

## 15. Para cerrar: tres ideas importantes

**1. Tipo de variable:** las medias, desviaciones estándar, cuantiles, histogramas y boxplots requieren una variable numérica apropiada; las categorías pueden servir para formar grupos.

**2. Interpretación:** un promedio no cuenta toda la historia. Conviene revisar mediana, dispersión, valores extremos y casos válidos.

**3. Visualización:** modificar `fill`, `width`, `binwidth`, `theme_*()` o `labs()` cambia la forma de presentar el análisis, no las respuestas de la encuesta.

**Pregunta final:** si dos grupos tienen la misma media de notas pero desviaciones estándar distintas, ¿qué nos dice eso sobre la distribución de sus notas?